# Low-Effort Optimization Playbook

The **six LOW-effort levers** — each a one-line change, a parameter tweak, or a short prompt refactor. No new infrastructure.

| # | Lever | What it does |
|---|---|---|
| 1 | Model selection | Match the model class to the task |
| 2 | Prompt design | Clear instructions + few-shot + structured output |
| 3 | Parameter tuning | `max_tokens`, `temperature`, `stop_sequences` |
| 4 | Prompt caching | Up to 90% off on repeated static content |
| 5 | Prompt engineering tricks | CoD, Self-Refine, Verbalized Sampling (+ managed Bedrock APO) |
| 6 | Adaptive thinking | Set an `effort` level; the model decides how much to think |

Each lever is independent — lift any one into your own application without the others. Run the setup cell, then work through them in order.

## Setup

In [ ]:
import boto3, time
from botocore.config import Config
from IPython.display import Markdown, display

from utils.pricing import calculate_actual_cost  # workshop helper

REGION = "us-east-1"
RUNTIME = boto3.client("bedrock-runtime", region_name=REGION)

# Workshop default models
HAIKU = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
SONNET = "global.anthropic.claude-sonnet-4-6"
# Adaptive thinking (cell below) runs on Opus 4.6 — the version available in the
# workshop sandbox account. Task Budgets (last cell) need Opus 4.7/4.8, so it
# keeps a separate 4.8 id and is off by default.
OPUS_4_6 = "global.anthropic.claude-opus-4-6-v1"
OPUS_4_8 = "global.anthropic.claude-opus-4-8"

# A neutral example task, reused only where a same-input comparison is useful.
# Each lever below is independent — swap in whatever input fits your own use case.
EXAMPLE_TASK = (
    "Summarize this product review in one sentence, then rate its sentiment "
    "as positive, neutral, or negative:\n\n"
    "'Delivery was fast and the packaging was solid, but the device stopped "
    "working after two weeks and support was slow to respond.'"
)


---

## 1. Model Selection — Lever 01

The highest-leverage cost decision is *which* model runs each call. The price gap between Claude tiers is **~5× per tier**, so a frontier model on a task a smaller one handles is pure waste — and usually slower too.

| Class | Model | Input / Output (per 1M) | Designed for |
|---|---|---|---|
| Small / fast | Haiku 4.5 | \$1 / \$5 | Classification, routing, extraction, short summaries |
| Mid-tier | Sonnet 4.6 | \$3 / \$15 | Tool use, multi-step reasoning — the agent workhorse |
| Frontier | Opus 4.8 | \$5 / \$25 | Complex coding, deep research, long-horizon agents |

The rule:

- **Start small, escalate on evidence.** Try Haiku; move up only when a scored eval shows it fails on *your* task.
- **Re-benchmark every model generation.** Today's small model ≈ last year's large one — Haiku 4.5 beats the prior Sonnet, so "needs Opus" decisions go stale.
- **Never decide by vibes.** One good (or bad) output isn't evidence; a held-out eval set is. Cheaper isn't free if quality drops.

Below: the same task on Haiku and Sonnet, with tokens, latency, and per-call cost side by side.

In [ ]:
def converse_simple(model_id, query, max_tokens=400):
    t0 = time.time()
    resp = RUNTIME.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": query}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": 0},
    )
    return {
        "text": resp["output"]["message"]["content"][0]["text"],
        "input_tokens": resp["usage"]["inputTokens"],
        "output_tokens": resp["usage"]["outputTokens"],
        "latency_ms": int((time.time() - t0) * 1000),
        "model": model_id,
    }


# The models reply in Markdown (headings, bold). We render each reply with
# display(Markdown(...)) so it shows formatted, not as raw "#"/"**" text.
rows = []
for m in [HAIKU, SONNET]:
    r = converse_simple(m, EXAMPLE_TASK)
    cost = calculate_actual_cost(r["input_tokens"], r["output_tokens"], r["model"])
    rows.append((m.split(".")[-1][:20], r["input_tokens"], r["output_tokens"], r["latency_ms"], cost))
    display(Markdown(f"### `{m}`"))
    display(Markdown(r["text"]))

summary = "### Summary\n\n| Model | In | Out | ms | $ |\n|---|---:|---:|---:|---:|\n"
for r in rows:
    summary += f"| {r[0]} | {r[1]} | {r[2]} | {r[3]} | {r[4]:.6f} |\n"
display(Markdown(summary))

**What you'll see**: for a well-defined, single-step task, Haiku 4.5 produces an answer indistinguishable from Sonnet 4.6 at ~3× lower cost and lower latency. Quality gaps only surface on ambiguous prompts — which is exactly why you confirm the cheaper choice on an eval set, not on one output.

---

## 2. Prompt Design — Lever 02

The most under-used lever, and the deepest. Most production prompts carry 30-50% wasted tokens — hedging, boilerplate, vague instructions — which costs money on *every* call **and** degrades quality. Three techniques, in order of how often you reach for them:

1. **Clear, specific instructions** — the baseline, always
2. **Few-shot examples** — when instructions alone leave room to guess
3. **Structured output** — when downstream code consumes the result

### 2.1 Clear, specific instructions

A well-designed prompt is **directive** (says exactly what to do, leading with the verb), **sectioned** (structure the model can follow), and **parsable** (output your code can consume without regex gymnastics). The rewrite below removes hedging, states the format upfront, and constrains the sentiment to an enum — roughly half the input tokens *and* more reliable output.

<div class="alert alert-block alert-info">
<b>Lead with the task, not the persona.</b> "You are a helpful assistant who…" is tokens of nothing for most tasks. Start with the verb; when code consumes the output, constrain the shape with a schema rather than over-specifying format in prose.
</div>

In [ ]:
EMAIL = (
    "Hi, the wireless headphones I ordered last week arrived with a cracked case "
    "and the left earbud won't charge. I'd like a replacement before the weekend."
)

VAGUE_PROMPT = (
    "Could you please help me by reading the customer's email below and then telling me "
    "what you think the main issue might be, and also what their sentiment seems to be, "
    "and what action you would recommend that we take?\n\n"
    f"Customer email: {EMAIL}"
)

STRUCTURED_PROMPT = (
    "Classify the email below.\n\n"
    f"Email: {EMAIL}\n\n"
    'Return JSON: {"issue": string, "sentiment": "positive"|"neutral"|"negative", "action": string}'
)

# Render each reply as Markdown: the VAGUE answer is long free-form prose; the
# STRUCTURED answer is a compact JSON block — the contrast is the whole point.
for label, prompt in [("VAGUE", VAGUE_PROMPT), ("STRUCTURED", STRUCTURED_PROMPT)]:
    r = converse_simple(HAIKU, prompt, max_tokens=200)
    display(Markdown(f"### {label} &mdash; in={r['input_tokens']} &middot; out={r['output_tokens']} tokens"))
    display(Markdown(r["text"]))

**What you'll see**: the structured prompt runs ~55% fewer input tokens than the vague one *and* returns tighter, parseable output. Cheaper and better aren't always a trade-off — here they're the same edit.

### 2.2 Few-shot examples

When instructions alone leave room to guess, examples resolve it faster than more prose — they're the most reliable way to pin **format**, tone, and how to handle borderline cases. They cost tokens on *every* request, so use them where they earn it and **cache them** (Lever 04 — a fixed example block is a textbook stable prefix).

**How many?** Start at zero, stop around five — the returns are front-loaded:

| Shots | What it buys |
|---|---|
| 0 | Baseline — always try first; fine when the task is unambiguous |
| 1 | The biggest single jump — pins down output **format** |
| 3 | Resolves tone + borderline cases — most production classification |
| 5 | Marginal polish; diminishing returns begin |
| >5 | Usually pays tokens for little gain (risk: overfitting to example surface form) |

**Which examples?** Pick **boundary cases** (the ones a human would hesitate on), keep them **diverse**, and for classification keep them **class-balanced**. A well-replicated finding (Min et al., 2022): the **format** of examples matters more than the labels being perfect — so keep every example byte-identical in structure. The cell below compares zero-shot vs few-shot across three inputs to show the consistency gain.

In [ ]:
# Zero-shot vs few-shot: extract product info into one exact pipe-delimited format.
# Run across several inputs to see CONSISTENCY, not just one lucky output.
PRODUCTS = [
    "Apple MacBook Pro 16-inch with M3 chip, 32GB RAM, Space Black - $2,499",
    "Sony WH-1000XM5 wireless noise cancelling headphones in silver for $348",
    "Samsung 65 inch OLED 4K Smart TV (2024 model) priced at $1,799.99",
]

ZERO_SHOT = """Extract product info in this exact format:
PRODUCT|BRAND|PRICE|CATEGORY

Product: "{product}"
Output:"""

# 3 boundary/diverse, class-balanced, byte-identical examples
FEW_SHOT = """Extract product info in this exact format:
PRODUCT|BRAND|PRICE|CATEGORY

Examples:
Product: "Dell XPS 15 laptop with Intel i7, 16GB memory - $1,299"
Output: XPS 15|Dell|1299|laptop
Product: "Bose QuietComfort earbuds, noise cancelling, $279 retail"
Output: QuietComfort Earbuds|Bose|279|audio
Product: "LG 55\\" C3 OLED TV (2023) on sale for $1,196.99"
Output: C3 OLED 55"|LG|1196.99|tv

Now extract:
Product: "{product}"
Output:"""

# For each product we print the zero-shot output then the few-shot output, so you can
# compare them line by line. Watch the few-shot column stay in ONE consistent format.
for p in PRODUCTS:
    outs = {}
    for label, tmpl in [("zero", ZERO_SHOT), ("few", FEW_SHOT)]:
        r = converse_simple(HAIKU, tmpl.format(product=p), max_tokens=60)
        outs[label] = r["text"].strip().splitlines()[0]
    print(f"\n{p[:60]}")
    print(f"  {'zero-shot:':<12}{outs['zero']}")
    print(f"  {'few-shot:':<12}{outs['few']}")

**What you'll see**: zero-shot often varies its format across the three inputs (extra prose, different field order, units left in the price). Few-shot pins all three to the exact `PRODUCT|BRAND|PRICE|CATEGORY` shape — the consistency that lets downstream code parse without special-casing.

### 2.3 Structured output — the highest-leverage reliability win

This is where teams lose time in production: the model returns *almost*-valid JSON, and your parser breaks on the one call in fifty that wrapped it in prose or hallucinated a field. There are **three** ways to get structure, in increasing order of guarantee. All three below extract the same `{issue, sentiment, action}` shape from the same email, so you can compare them directly.

**(a) Prompt-based** — you *ask* for a format. Cheapest, no setup, but the model can still drift (stray prose, markdown fences, missing keys). Fine for prototypes and human-read output.

In [ ]:
import json

CLASSIFY_EMAIL = (
    "My order #A-123 arrived with a cracked screen and the charger is missing. "
    "I want a replacement this week."
)

# (a) Prompt-based: ask for raw JSON. temperature=0 helps, but guarantees nothing.
resp = RUNTIME.converse(
    modelId=SONNET,
    system=[{"text": 'Classify the email. Reply with ONLY raw JSON, no markdown, no prose: '
                     '{"issue": string, "sentiment": "positive"|"neutral"|"negative", "action": string}'}],
    messages=[{"role": "user", "content": [{"text": CLASSIFY_EMAIL}]}],
    inferenceConfig={"maxTokens": 256, "temperature": 0},
)
text = resp["output"]["message"]["content"][0]["text"]
data = json.loads(text)  # may raise if the model wrapped it in ```json fences or added prose
print("(a) prompt-based:")
print(json.dumps(data, indent=2))

**The prefill trick (model-dependent).** A classic nudge is to *prefill the assistant turn* with a single `{` — the model continues from that token, forcing JSON onset and skipping any "Sure, here's the JSON:" preamble.

<div class="alert alert-block alert-warning">
<b>Prefill is rejected by the thinking-capable models.</b> On Opus 4.6/4.7/4.8 and Sonnet 4.6 it returns <code>ValidationException: This model does not support assistant message prefill</code> (and it's incompatible with adaptive thinking, which must emit a reasoning block first). It still works on non-thinking models like <b>Haiku 4.5</b> — so prefill is a Haiku-tier trick, not universal. On Opus/Sonnet, use native structured output (c) instead.
</div>

In [ ]:
# Prefill works on NON-thinking models (Haiku 4.5). End messages with an assistant "{".
resp = RUNTIME.converse(
    modelId=HAIKU,
    system=[{"text": 'Reply with ONLY raw JSON: '
                     '{"issue": string, "sentiment": "positive"|"neutral"|"negative", "action": string}'}],
    messages=[
        {"role": "user", "content": [{"text": CLASSIFY_EMAIL}]},
        {"role": "assistant", "content": [{"text": "{"}]},   # ← prefill forces the JSON onset
    ],
    inferenceConfig={"maxTokens": 256, "temperature": 0},
)
data = json.loads("{" + resp["output"]["message"]["content"][0]["text"])
print("(a') Haiku prefill:")
print(json.dumps(data, indent=2))

**(b) Tool use / function calling** — you define a tool with an input schema; the model "calls" it and Bedrock returns arguments **conforming to that schema**. It literally cannot emit a shape that doesn't match — the schema is the contract it fills. This is the right default when the model is genuinely **calling a function** in an agent loop. The cost: tool definitions add input tokens on every call.

In [ ]:
# (b) Tool use: schema-enforced output via a forced tool call.
SCHEMA = {
    "type": "object",
    "properties": {
        "issue": {"type": "string"},
        "sentiment": {"type": "string", "enum": ["positive", "neutral", "negative"]},
        "action": {"type": "string"},
    },
    "required": ["issue", "sentiment", "action"],
    "additionalProperties": False,
}
tool = {"toolSpec": {
    "name": "classify_email",
    "description": "Classify a customer email.",
    "inputSchema": {"json": SCHEMA},
}}
resp = RUNTIME.converse(
    modelId=SONNET,
    messages=[{"role": "user", "content": [{"text": CLASSIFY_EMAIL}]}],
    toolConfig={"tools": [tool], "toolChoice": {"tool": {"name": "classify_email"}}},  # force the call
    inferenceConfig={"maxTokens": 256, "temperature": 0},
)
# Pull the structured args straight out of the toolUse block (already a dict — no parsing).
data = next(c["toolUse"]["input"] for c in resp["output"]["message"]["content"] if "toolUse" in c)
print("(b) tool-use:")
print(json.dumps(data, indent=2))
print("    input tokens:", resp["usage"]["inputTokens"], "(higher — the schema rides along)")

**(c) Native structured output / JSON Schema** — the strictest contract, and the one people conflate with tool use. The distinction: **tool use (b)** dresses your schema as a *fake function* and you dig the result out of a `toolUse` block; **native (c)** attaches the schema to the request and Bedrock uses **constrained decoding** so the model's *normal text answer* is guaranteed schema-valid — no fake tool, the JSON lands in the regular text block. Reach for (c) when the goal is simply "give me clean JSON in this shape"; reach for (b) when the model is actually calling a function.

<div class="alert alert-block alert-warning">
<b>Two gotchas.</b> On Converse the schema must be a JSON <i>string</i> (<code>json.dumps</code>) — a raw dict 400s (on <code>InvokeModel</code> it's the opposite: a real object). And native structured output is <b>not</b> available on the <code>bedrock-mantle</code> path — use it on <code>bedrock-runtime</code> (Converse/InvokeModel), which is what this workshop uses. Every object needs <code>additionalProperties: false</code>; the first call with a new schema compiles a grammar (cached ~24h).
</div>

In [ ]:
# (c) Native structured output: schema goes in outputConfig.textFormat, as a JSON STRING.
resp = RUNTIME.converse(
    modelId=SONNET,
    messages=[{"role": "user", "content": [{"text": CLASSIFY_EMAIL}]}],
    inferenceConfig={"maxTokens": 256, "temperature": 0},
    outputConfig={"textFormat": {
        "type": "json_schema",
        "structure": {"jsonSchema": {
            "name": "classify_email",
            "description": "Classify a customer email.",
            "schema": json.dumps(SCHEMA),   # note: a JSON string, not a dict
        }},
    }},
)
data = json.loads(resp["output"]["message"]["content"][0]["text"])  # guaranteed schema-valid
print("(c) native JSON schema:")
print(json.dumps(data, indent=2))
print("    input tokens:", resp["usage"]["inputTokens"], "(lower than tool use — no tool definition)")

**What you'll see**: all three return the same `{issue, sentiment, action}` shape, but with different guarantees and costs.

| Approach | Guarantee | Input-token cost | Reach for it when |
|---|---|---|---|
| (a) Prompt-based | Low — model may drift | Lowest | Prototypes, simple shapes, human reads the output |
| (b) Tool use | High — schema-enforced via the call | Higher (schema rides along) | The model is genuinely calling a **function** in an agent loop |
| (c) Native JSON Schema | Highest — API-validated | Lower than tool use | "Give me clean JSON in this shape" — pure data extraction (**default**) |

**The take-home**: if downstream code consumes the output, don't *ask* for JSON — *constrain* it. Use native structured output for data shaping, tool use when the model is actually calling a function. The cost is the same; the difference is whether your pipeline breaks at 2am on the one malformed response.

---

## 3. Parameter Tuning — Lever 03

Four inference-config values that ship in minutes and cost nothing to change — but the defaults favor "won't surprise you," not "fast, cheap, predictable." Getting them right buys 10-30% on latency and concurrency before you touch a prompt.

- **`max_tokens`** — the load-bearing one. On Bedrock it sets the **quota reserved** up front (output burns at **5× for Claude 3.7+**), refunded after. `max_tokens=4096` on a 200-token reply reserves ~20× the quota you'll use, throttling concurrency under load. Per-call *cost* is unchanged (you pay for tokens generated); *concurrency* is not. Right-size it.
- **`temperature`** — `0` for classification, extraction, structured output (deterministic); `0.3-0.7` for generation; higher for brainstorming. The API default leans high — don't ship it for production tasks.
- **`stop_sequences`** — a free latency win: generation halts the instant a known terminator appears, shaving the unused tail off time-to-last-token.

<div class="alert alert-block alert-warning">
<b><code>max_tokens</code> is "absolute max," not "max desired."</b> Set it tight; raise it only if you see truncation. <code>stop_sequences</code> only helps with a real terminator — don't invent fragile ones.
</div>

In [ ]:
# max_tokens=4096 vs max_tokens=200 yields identical output (the model decides when to
# stop), but the TPM quota *reserved* differs 20x. Show the actual output first (rendered
# as Markdown), then the reserved-TPM table.
rows = []
answer = None
for mt in [4096, 200]:
    r = converse_simple(HAIKU, EXAMPLE_TASK, max_tokens=mt)
    answer = r["text"]  # identical for both settings
    reserved_tpm = r["input_tokens"] + (mt * 5)  # 5x burndown for newer Claude
    rows.append((mt, r["output_tokens"], reserved_tpm))

md = f"**INPUT**\n\n> {EXAMPLE_TASK.replace(chr(10), chr(10) + '> ')}\n\n"
md += f"**OUTPUT** (identical regardless of `max_tokens`):\n\n{answer}\n\n"
md += "**Reserved quota**\n\n| max_tokens | out | reserved TPM |\n|---:|---:|---:|\n"
for mt, out, tpm in rows:
    md += f"| {mt} | {out} | {tpm:,} |\n"
display(Markdown(md))

In [ ]:
# 2. stop_sequences — early termination on a known boundary. Show the actual reply so you
#    can see it stop before "###STOP" is emitted (output ends earlier, fewer tokens, faster).
structured_prompt = (
    "Categorize the support email into one word (shipping, product, billing, other). "
    "Reply with the category, then ###STOP\n\n"
    f"Email: {EMAIL}\n\n"
    "Category: "
)

for use_stop in [False, True]:
    cfg = {"maxTokens": 200, "temperature": 0}
    if use_stop:
        cfg["stopSequences"] = ["###STOP"]
    t0 = time.time()
    resp = RUNTIME.converse(
        modelId=HAIKU,
        messages=[{"role": "user", "content": [{"text": structured_prompt}]}],
        inferenceConfig=cfg,
    )
    text = resp["output"]["message"]["content"][0]["text"]
    print(f"\nstop_sequences={use_stop}")
    print(f"  output: {text!r}")
    print(f"  out_tokens={resp['usage']['outputTokens']}  ms={int((time.time()-t0)*1000)}")

In [ ]:
# 3. temperature — run the same creative prompt 3x at each setting to see consistency vs variation.
TAGLINE = "Write a one-sentence tagline for a coffee shop."

for temp in [0.0, 0.7, 1.0]:
    print(f"temperature={temp}")
    for run in range(3):
        resp = RUNTIME.converse(
            modelId=HAIKU,
            messages=[{"role": "user", "content": [{"text": TAGLINE}]}],
            inferenceConfig={"maxTokens": 40, "temperature": temp},
        )
        print(f"  run {run + 1}: {resp['output']['message']['content'][0]['text'].strip()}")
    print()

**What you'll see**: (1) both `max_tokens` settings yield the same output — the model decides when to stop — but the reserved TPM differs ~20×, the concurrency you're silently giving up. (2) With `stop_sequences` on, `out_tokens` drops the moment `###STOP` appears and latency falls with it. (3) At `temperature=0` the three runs are near-identical; at `0.7`–`1.0` they diverge — which is why classification/extraction use `0` and brainstorming uses higher.

---

## 4. Prompt Caching — Lever 04

Most requests resend the same system prompt and tool definitions every time, paying full input price for tokens that never change. Caching pays full price **once** (the write, billed at 1.25×), then **0.1×** for every read — break-even is ~3 reads, after which it's pure savings.

- **Bedrock caches explicitly for Claude.** *You* place a `cachePoint` marker; no marker means nothing is cached (no implicit fallback like OpenAI/Gemini).
- **The cache key is the longest token prefix from position 0**, because Bedrock assembles requests in a fixed order (tools → system → messages). So the rule is **static first, dynamic last**: stable content (tool schemas, system prompt, few-shot examples) before the marker; the volatile tail (user message, timestamps) after.
- **You can cache the conversation too**, not just the static prefix — keep history append-only and move the `cachePoint` to the end of each turn. Re-summarizing earlier turns rewrites the prefix and busts the whole cache.

The demo below caches a real **product manual** (`data/product_manual.txt`) as the static prefix and asks two different questions against it — so call 2 reads the cached manual instead of re-paying for it.

<div class="alert alert-block alert-warning">
Caching is <b>exact-prefix</b> — there's no "close enough." Edit one word inside the cached region and the next call is a full miss that pays the write premium again. Watch <b>cache hit rate</b> like an SLO (target ≥70%). Sonnet 4.6 has the lowest minimum (1,024 tokens); Haiku 4.5 and Opus 4.8 need 4,096.
</div>

In [ ]:
# A real, large, stable document — the product manual — is the cached prefix.
# It's ~1.9K tokens, over the 1,024 minimum for Sonnet 4.6.
with open("data/product_manual.txt") as f:
    PRODUCT_MANUAL = f.read()
display(Markdown(f"Loaded `product_manual.txt`: **{len(PRODUCT_MANUAL):,} chars**"))


def ask_with_cache(question, cache=True):
    # Static manual first (cached), cachePoint marker, then the volatile question.
    content = [{"text": f"Use the product manual below to answer questions.\n\nMANUAL:\n{PRODUCT_MANUAL}"}]
    if cache:
        content.append({"cachePoint": {"type": "default"}})
    content.append({"text": f"\n\nQUESTION: {question}"})

    resp = RUNTIME.converse(
        modelId=SONNET,
        messages=[{"role": "user", "content": content}],
        inferenceConfig={"maxTokens": 200, "temperature": 0},
    )
    u = resp["usage"]
    return {
        "answer": resp["output"]["message"]["content"][0]["text"],
        "input": u.get("inputTokens", 0),
        "cache_read": u.get("cacheReadInputTokens", 0),
        "cache_write": u.get("cacheWriteInputTokens", 0),
        "output": u.get("outputTokens", 0),
    }


def show(call_label, question):
    r = ask_with_cache(question)
    usage = {k: r[k] for k in ("input", "cache_read", "cache_write", "output")}
    display(Markdown(
        f"**{call_label}**\n\n"
        f"**Q:** {question}\n\n"
        f"**A:**\n\n{r['answer']}\n\n"
        f"`usage = {usage}`"
    ))


show("Call 1 (cache write expected — first time the manual is seen)",
     "What's the return window for a defective item?")
show("Call 2 (cache read expected — same manual, different question)",
     "Is the warranty transferable to a new owner?")

**What you'll see**: on call 2, `cache_read` jumps to roughly the manual's token count while `input` drops to just the question — the same stable prefix served at 0.1× instead of 1×. Call 1 shows the one-time `cache_write` premium instead.

---

## 5. Prompt Engineering Tricks — Lever 05

A handful of research-backed prompting patterns — **tools, not requirements**. A clean prompt on the right model (Levers 01-02) beats a clever trick on a sloppy one, so reach for these only when a specific problem calls for it:

| Technique | Use it when | Trade-off |
|---|---|---|
| **Chain-of-Draft** | You want CoT-quality reasoning but the token/latency bill hurts | Slight quality drop on the very hardest problems |
| **Self-Refine** | Quality matters more than throughput; want a visible critique step | 2-3× the call cost |
| **Verbalized Sampling** | Outputs all "sound the same" and you need diversity | A few extra tokens per call |
| **Chain-of-Verification** | Fact-critical output where hallucinations are unacceptable | 4-5× single-call cost |
| **Bedrock APO** (managed) | You have eval data and want the rewrite found *for* you, or are migrating a prompt to a new model | Async eval-driven job (~20-50 min) — reference code, not a live cell |

We run the two with the clearest live cost stories below — **Chain-of-Draft** and **Self-Refine** — then point to **Bedrock APO** as the managed, eval-driven escalation.

**Chain-of-Draft (CoD)** keeps CoT's step-by-step decomposition but caps each step at **~5 words**, so reasoning tokens drop **50-70%** while accuracy holds — because most of CoT's benefit is the *act* of decomposing, not the prose.

<div class="alert alert-block alert-info">
<b>Reasoning models subsume part of this.</b> On a thinking-capable model (Lever 06), the model already runs an internal generate-critique loop — so reach for explicit Self-Refine/CoVe when you're on a non-thinking model (Haiku) or you specifically need the critique <i>visible</i> (human review, logging why an answer changed). <b>Bedrock APO</b> is the managed escalation when hand-tuning stops scaling.
</div>

In [ ]:
REASONING_QUERY = (
    "A customer requests a refund on a monitor bought 45 days ago with a dead pixel. "
    "Policy: full refund within 30 days; manufacturer warranty for defects from 30-365 days. "
    "What's the right answer? Walk through your reasoning, then give the final recommendation."
)

COT_PROMPT = REASONING_QUERY + "\n\nThink step by step before answering."

COD_PROMPT = (
    REASONING_QUERY + "\n\nThink step by step, but only keep a minimum draft for each thinking step, "
    "with 5 words at most. Return the answer at the end after a separator ####."
)

# Chain-of-Thought (CoT) reasons in full sentences; Chain-of-Draft (CoD) keeps each step
# to a terse 5-word draft — same final answer, far fewer output tokens. We render each
# model reply as Markdown (it returns headings/bullets) so it's readable, not raw "#" text.
for label, prompt in [("Chain-of-Thought (CoT)", COT_PROMPT), ("Chain-of-Draft (CoD)", COD_PROMPT)]:
    r = converse_simple(HAIKU, prompt, max_tokens=600)
    display(Markdown(f"### {label} &mdash; in={r['input_tokens']} &nbsp; out={r['output_tokens']} tokens"))
    display(Markdown(r["text"]))

**What you'll see**: CoD's `output_tokens` should be roughly half of CoT's while the final recommendation stays the same — same reasoning quality, half the cost.

**Self-Refine** is a different shape: generate → critique → revise, as three calls. The trick is that **naming the flaw before fixing it** beats a blind second draft — so the critique call is *forbidden from rewriting*. The cost lever is right there in the model choice: a cheap model generates and revises, a stronger model critiques. The cell below runs all three steps so the critique is **visible**.

In [ ]:
# Self-Refine: generate (Haiku) -> critique (Sonnet, NO rewrite) -> revise (Haiku).
# Each model reply is rendered as Markdown so the formatting is readable, not raw "#" text.

def chat(model_id, prompt, max_tokens=300):
    r = RUNTIME.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
        inferenceConfig={"maxTokens": max_tokens, "temperature": 0},
    )
    return r["output"]["message"]["content"][0]["text"].strip()

TASK = 'Write a support reply to: "I was charged twice for October. Fix it."'

# Step 1 - Generate (cheap model)
draft = chat(HAIKU, TASK)
display(Markdown("### Step 1 &mdash; Draft (Haiku generates)"))
display(Markdown(draft))

# Step 2 - Critique (stronger model; say what's wrong, do NOT rewrite)
critique = chat(SONNET, (
    f"Here is a draft support reply:\n{draft}\n\n"
    "Critique it, one bullet each: empathy · concrete next step · accuracy "
    "(promise nothing we can't verify) · tone. Say what's wrong; do NOT rewrite."
))
display(Markdown("### Step 2 &mdash; Critique (Sonnet finds flaws, no rewrite)"))
display(Markdown(critique))

# Step 3 - Revise (cheap model can follow a clear critique)
final = chat(HAIKU, (
    f"Draft:\n{draft}\n\nCritique:\n{critique}\n\n"
    "Rewrite to address every point; keep what the critique said was already good."
))
display(Markdown("### Step 3 &mdash; Revised (Haiku applies the critique)"))
display(Markdown(final))

**What you'll see**: the Step-2 critique names concrete flaws (a vague next step, an unverifiable promise) without rewriting, and Step 3 produces a visibly better reply that addresses each point. That visible critique is the whole reason to reach for Self-Refine over a thinking model — you can log *why* the answer changed. The trade-off is real: this is 3× the calls of a single generation.

### Automate it: Bedrock Advanced Prompt Optimization (APO)

Hand-tuning is where everyone starts, but it doesn't scale and doesn't transfer cleanly when you change models. **Amazon Bedrock Advanced Prompt Optimization (APO)** is the managed escalation: you give it prompt templates, example inputs, optional ground-truth answers, and an **evaluation metric** (a Lambda scorer, LLM-as-a-Judge, or steering criteria), and it runs a feedback loop that rewrites the prompts — comparing your original against optimized versions across **up to 5 models at once**. It does two jobs: **optimize** a prompt against your metric, and **migrate** a prompt to a different model (e.g. moving down to a cheaper one) verifying no regression *before* you switch — which pairs directly with **Lever 01: Model Selection** above.

It's an **asynchronous job** (`bedrock.create_advanced_prompt_optimization_job`, ~20-50 min), so it runs from the console/API rather than a live workshop cell. The official runnable sample — Lambda-metric, LLM-as-judge, and steering-criteria notebooks — is here:

<div class="alert alert-block alert-info">

**Reference:** [aws-samples/amazon-bedrock-samples → `advanced-prompt-optimization`](https://github.com/aws-samples/amazon-bedrock-samples/tree/main/advanced-prompt-optimization) — three notebooks (`01_lambda_metric`, `02_llm_as_judge`, `03_steering_criteria`) plus a reference `utils.py`. Submit with `create_advanced_prompt_optimization_job`, poll with `get_advanced_prompt_optimization_job`.

</div>

For programmatic, multi-stage pipeline optimization with full control over the algorithm, that's the open-source **GEPA / DSPy** route — [`03-high-effort.ipynb`](./03-high-effort.ipynb), Lever 15.

---

## 6. Adaptive Thinking — Lever 06

"Thinking" lets a Claude model spend extra internal tokens reasoning before it answers — better on hard reasoning, coding, and planning, at the cost of more output tokens and latency. The old control was a fixed `budget_tokens` you tuned per prompt. **Adaptive thinking** replaces it: set an `effort` level and the model decides, per request, how much to think — both a *quality* lever (think when it helps) and a *cost/latency* lever (don't pay when it doesn't).

- **One field, same Converse API** — add `thinking` and `output_config` to `additionalModelRequestFields`. No new endpoint, no SDK swap.
- **Four levels**: `low`, `medium`, `high` (default), `max`. The practical pattern is to *step down* from `high` once evals show a lower level holds quality.
- **Model support**: Opus 4.6/4.7/4.8 and Sonnet 4.6 — **not** Haiku 4.5. On Opus 4.7/4.8 it's the *only* thinking mode (manual `budget_tokens` returns a 400).

<div class="alert alert-block alert-warning">
<b>Placement matters.</b> <code>effort</code> goes in a <i>separate</i> <code>output_config</code> object, <b>not</b> inside <code>thinking</code> — putting it inside returns a <code>ValidationException</code>. The cell below sweeps <code>low → medium → high</code> on Opus 4.6 against one ambiguous ticket. (We use Opus 4.6 — the version available in the workshop sandbox; the lever behaves identically on 4.7/4.8, the production default.)
</div>

In [ ]:
AMBIGUOUS_TICKET = (
    "A customer says they bought a laptop 'a couple of months ago', the screen 'started doing something weird' "
    "after their kid dropped it once, and they want 'whatever you can do for me'. "
    "Apply policy. Decide the right resolution. Justify briefly."
)

# Adaptive thinking is a one-field change on the SAME Converse API (bedrock-runtime) —
# `thinking` and `output_config` go inside additionalModelRequestFields. No Mantle needed.
# Claude 4+ returns SUMMARIZED thinking: reasoningContent.reasoningText carries
#   .text       -> a human-readable summary of the reasoning (shown below)
#   .signature  -> a cryptographic proof reasoning occurred (not human-readable)
rows = []
for effort in ["low", "medium", "high"]:
    t0 = time.time()
    resp = RUNTIME.converse(
        modelId=OPUS_4_6,
        messages=[{"role": "user", "content": [{"text": AMBIGUOUS_TICKET}]}],
        inferenceConfig={"maxTokens": 1024},
        additionalModelRequestFields={
            "thinking": {"type": "adaptive"},
            "output_config": {"effort": effort},   # low | medium | high (default) | max
        },
    )
    ms = int((time.time() - t0) * 1000)
    thinking, answer = "", ""
    for b in resp["output"]["message"]["content"]:
        if "reasoningContent" in b:
            thinking = b["reasoningContent"]["reasoningText"]["text"]
        elif "text" in b:
            answer = b["text"]
    u = resp["usage"]
    rows.append((effort, bool(thinking), u["outputTokens"], ms))

    md = f"### effort = `{effort}` &mdash; out={u['outputTokens']} tokens &middot; {ms} ms\n\n"
    if thinking:
        quoted = "\n".join("> " + line for line in thinking.splitlines())
        md += f"**Summarized thinking** ({len(thinking)} chars):\n\n{quoted}\n\n"
    else:
        md += "_(model chose to skip thinking on this query — adaptive thinking deciding for you)_\n\n"
    md += f"**Answer:**\n\n{answer}"
    display(Markdown(md))

# Compact recap: thinking grows with effort, so output tokens and latency climb.
recap = "### Effort sweep recap\n\n| effort | thought | out_tokens | ms |\n|---|:---:|---:|---:|\n"
for effort, thought, out, ms in rows:
    recap += f"| {effort} | {'yes' if thought else 'no'} | {out} | {ms} |\n"
display(Markdown(recap))

**What you'll see**: as `effort` rises `low`→`high`, the model thinks more — output tokens climb (thinking bills as output) and both the reasoning and the final answer grow more thorough. At `effort=low` on a query it deems simple the model may skip thinking entirely — that's the adaptive part: the model, not you, decides whether to think.

<div class="alert alert-block alert-info">
<b>Summarized thinking — and where to read it.</b> Claude 4+ returns a <i>summarized</i> version of its reasoning, not the verbatim internal chain. On Converse it arrives in the <code>reasoningContent</code> block as <code>reasoningText.text</code> (the human-readable summary, printed above each answer) plus <code>reasoningText.signature</code> (a cryptographic proof that reasoning happened, not meant to be read). So you <i>can</i> see the summary, but token/latency cost is still the metric that matters — watch output tokens climb with effort.
</div>

### Bonus: Task Budgets (beta) — cap the whole agentic loop

`effort` controls one call; **Task Budgets** cap token spend across an entire agentic loop, so the model winds down gracefully instead of getting cut off. It's gated by the beta header `anthropic_beta: ["task-budgets-2026-03-13"]` — present it and it works on Converse, InvokeModel, *or* Mantle (omit it → `ValidationException: output_config.task_budget: Extra inputs are not permitted`). `total` is a *soft* budget; `max_tokens` is still the hard per-request ceiling.

<div class="alert alert-block alert-info">
Task Budgets currently need Opus 4.7 or 4.8, and the Workshop Studio sandbox has restricted access to those models, so the next cell won't run here as-is. We've left it set to <code>RUN_TASK_BUDGETS = False</code> so the notebook runs end-to-end — the code is there to read, and you can flip it to <code>True</code> on an account that has Opus 4.7/4.8 access to try it yourself.
</div>

In [ ]:
# Task Budgets cap token spend across a whole agentic loop (beta): add the beta header
# + output_config.task_budget on the same Converse API. They require Opus 4.7/4.8 (Opus 4.6
# rejects them), which the workshop sandbox doesn't have — so this cell is OFF by default.
# Flip RUN_TASK_BUDGETS = True on an account with Opus 4.7/4.8 access to run it.
RUN_TASK_BUDGETS = False

if RUN_TASK_BUDGETS:
    resp = RUNTIME.converse(
        modelId=OPUS_4_8,
        messages=[{"role": "user", "content": [{"text":
            "Plan a comprehensive market-research project for a fintech startup. Be thorough."}]}],
        inferenceConfig={"maxTokens": 1024},
        additionalModelRequestFields={
            "thinking": {"type": "adaptive"},
            "anthropic_beta": ["task-budgets-2026-03-13"],          # the gate — omit it and the call 400s
            "output_config": {"effort": "high", "task_budget": {"type": "tokens", "total": 25000}},
        },
    )
    u = resp["usage"]
    print(f"stop={resp['stopReason']}  in={u['inputTokens']}  out={u['outputTokens']}")
    print(resp["output"]["message"]["content"][-1].get("text", "")[:400])
else:
    print("Task Budgets demo is OFF (needs Opus 4.7/4.8 access). Set RUN_TASK_BUDGETS = True to run.")


## What you've done

You've applied all six LOW-effort levers, each a standalone technique:

1. **Model selection** — match the model class to the task; escalate on evidence
2. **Prompt design** — clear instructions, few-shot for ambiguity, structured output (prompt-based / tool-use / native schema)
3. **Parameter tuning** — right-size `max_tokens`, `temperature=0`, `stop_sequences`
4. **Prompt caching** — cache the stable prefix for up to 90% off the repeated tokens
5. **Prompt engineering tricks** — CoD and Self-Refine where they fit; Bedrock APO (managed, eval-driven) when hand-tuning stops scaling
6. **Adaptive thinking** — let the model manage reasoning depth with `effort`

Each is independent — lift any one into your own application on its own. Stacked where they fit, they typically yield 30-50% cost reduction with no quality regression.

**Next**: [`02-medium-effort.ipynb`](./02-medium-effort.ipynb) for the architectural moves.